# Estimating Extreme Credit Portfolio Risk with Importance Sampling

This notebook asks whether systemic-factor importance sampling can reduce the sampling variance of 99% expected shortfall (ES99) while preserving the point estimates produced by plain Monte Carlo. Core calculations are imported from the repository's Python modules; this notebook contains no duplicate model implementation.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().resolve()
if not (ROOT / 'requirements.txt').is_file():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from config import AnalysisConfig
from experiment import run_analysis

INPUT = ROOT / 'data/raw/issuer_portfolios_lqd_hyg.xlsx'
OUTPUT = ROOT / 'outputs'
pd.options.display.float_format = '{:,.2f}'.format

## Portfolios and issuer data

The workbook supplies three 50-issuer portfolios: 30% LQD / 70% HYG, 50% / 50%, and 70% / 30%. LQD represents investment-grade corporate credit and HYG represents high-yield credit. `Sheet1`, `Sheet2`, and `Sheet3` contain computed issuer names, exposure weights, and one-year default probabilities. The data module validates all fields and uses each supplied weight directly as $EAD_i=w_iV$.

## One-factor Gaussian credit model

For issuer $i$, $X_i=\sqrt{\rho}Z+\sqrt{1-\rho}\varepsilon_i$ and default occurs when $X_i<\Phi^{-1}(PD_i)$. A path loss is $L=\sum_i EAD_i\,LGD\,\mathbf 1\{X_i<c_i\}$. The baseline assumes a $100 million portfolio, 40% LGD, 20% asset correlation, and a one-year default-only horizon.

## Plain Monte Carlo and risk measures

Plain Monte Carlo draws $Z$ and all issuer shocks from independent standard normals. EL is the sample mean; VaR uses NumPy's empirical quantile; ES is the mean of every observed loss satisfying $L\geq VaR$. The inclusive tail matters because portfolio losses are discrete.

## Importance sampling, weighted tails, and ESS

IS draws $Z\sim N(\mu_{IS},1)$ with $\mu_{IS}<0$ and leaves idiosyncratic shocks unchanged. Each path receives $w(Z)=\exp(-\mu_{IS}Z+\tfrac12\mu_{IS}^2)$. EL and tail averages use self-normalized weights. Weighted VaR is the smallest observed loss whose cumulative normalized weight reaches the requested probability. Kish $ESS=(\sum w)^2/\sum w^2$ reports the effective number of equally weighted paths.

## Selecting the systemic shift

For each portfolio, 50 plain replications estimate baseline ES99 variance. Every shift in $0,-0.1,\ldots,-1.4$ then receives 50 IS replications. Zero is excluded and the remaining shift with the largest plain-to-IS ES99 variance ratio is selected. This reproduces the original rule; an ESS constraint would be a useful robustness extension.

In [ ]:
summary_path = OUTPUT / 'tables/all_portfolios_summary.csv'
if not summary_path.is_file():
    results = run_analysis(INPUT, OUTPUT, AnalysisConfig())
else:
    results = {
        'selected_mu': pd.read_csv(OUTPUT / 'tables/selected_mu.csv'),
        'summary': pd.read_csv(summary_path),
    }
results['selected_mu']

## Main results

In [ ]:
main_metrics = results['summary'].query("Metric in ['EL', 'VaR99', 'ES99']")
main_metrics[['Portfolio', 'Method', 'Metric', 'Mean', 'SD', 'SE', 'Average_ESS', 'Selected_mu_IS']]

In [ ]:
for label in ('30_70', '50_50', '70_30'):
    display(Image(filename=OUTPUT / f'figures/{label}_mu_selection.png'))
    display(Image(filename=OUTPUT / f'figures/{label}_es99_scatter.png'))

## Interpretation

The selected negative shifts deliberately produce more adverse systemic scenarios. ES99 means remain close across methods, while IS replication clouds are visibly tighter. Variance-reduction ratios above one show the gain directly. ESS generally declines as the proposal becomes more aggressive, revealing the trade-off between observing more tail losses and concentrating likelihood-ratio weights.

## Original-notebook comparison

In [ ]:
comparison_path = OUTPUT / 'tables/original_vs_refactored.csv'
if comparison_path.is_file():
    display(pd.read_csv(comparison_path))
else:
    print('Run the documented baseline comparison to generate original_vs_refactored.csv.')

## Limitations and extensions

The analysis assumes homogeneous deterministic LGD, constant correlation, a static one-year horizon, Gaussian dependence, and default-only losses. Its IS estimators are self-normalized, the shift is simulation-selected, and neither parameter uncertainty nor model calibration is studied. Extensions include heterogeneous or stochastic LGD, sector factors, heavier-tailed copulas, adaptive or ESS-constrained IS, parameter uncertainty, and multi-period migration.